# Train the response-quality reward model

Fine-tunes a cross-encoder on our `(context, response)` pairs labeled with `goal_progress_assessments` from train conversations. The trained model powers the inference-time response reranker: at blind-time we sample K responses per query and pick the one the reward model says is most likely to move the user toward their goal.

**Base model**: `cross-encoder/ms-marco-MiniLM-L-6-v2` — 22M params, already fine-tuned for query-document scoring, very fast to fine-tune on T4.

**Training data**: run `scripts/build_reward_dataset.py` locally first to produce `data/reward_train.parquet`, then commit it (or upload via the Drive cell below).

**Target**: binary classification (MOVES_TOWARD_GOAL=1, DOES_NOT=0). Training loss = BCE. Primary eval metric = AUC on session-held-out validation slice.

**Wall time**: ~20–40 min on T4 for ~60k rows × 2 epochs. Model weights are ~90 MB.

Output → `/content/drive/MyDrive/recsys2026-reward-model/` (zip + raw weights).

In [ ]:
# 1) Verify GPU.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model so we can pull data/reward_train.parquet (if committed) + the helper scripts.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026-lora-tutorial
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026-lora-tutorial
%cd /content/recsys2026-lora-tutorial

print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%ndate:    %ai%nsubject: %s'
print()

In [ ]:
# 3) Install deps. Use requirements.txt first so the mcrs package (which
# eagerly loads bm25s + omegaconf + pandas through its __init__) can import
# cleanly when scripts/build_reward_dataset.py runs. Then add the
# reward-model-specific libs on top.
!pip install -q -r requirements.txt
!pip install -q sentence-transformers scipy scikit-learn pyarrow
!python -c "import bm25s, sentence_transformers, torch, transformers, omegaconf; print('bm25s', bm25s.__version__, 'sentence-transformers', sentence_transformers.__version__, 'torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 4) Build (or load) the training parquet.
# Option A: generate it from the HF train split directly on Colab (~1-2 min)
#           — use this the FIRST time or whenever schema changes.
# Option B: download an existing parquet from Drive if you already built it.
#
# Default: build it.
import os
if not os.path.isfile('data/reward_train.parquet'):
    !python scripts/build_reward_dataset.py --n-sessions 15000 --out data/reward_train.parquet --split-val 0.1
else:
    print('reward_train.parquet already present, skipping rebuild')
!ls -lh data/reward_train.parquet

In [ ]:
# 5) Load data + split.
import pandas as pd
df = pd.read_parquet('data/reward_train.parquet')
print(f'total rows: {len(df)}  positives: {int(df.label.sum())}')
train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df = df[df['split'] == 'val'].reset_index(drop=True)
print(f'train: {len(train_df)}  val: {len(val_df)}')
print(f'train pos rate: {train_df.label.mean():.3f}  val pos rate: {val_df.label.mean():.3f}')

In [ ]:
# 6) Fine-tune cross-encoder. BCE loss, 2 epochs, batch 64, lr 2e-5.
from sentence_transformers import CrossEncoder
from sentence_transformers.readers import InputExample
from torch.utils.data import DataLoader
import torch

BASE = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
OUT = '/content/reward_model'

train_examples = [
    InputExample(texts=[r.text_a, r.text_b], label=float(r.label))
    for r in train_df.itertuples(index=False)
]
val_examples = [
    InputExample(texts=[r.text_a, r.text_b], label=float(r.label))
    for r in val_df.itertuples(index=False)
]

model = CrossEncoder(BASE, num_labels=1, max_length=384, device='cuda' if torch.cuda.is_available() else 'cpu')
train_loader = DataLoader(train_examples, shuffle=True, batch_size=64)

from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
val_evaluator = CEBinaryClassificationEvaluator.from_input_examples(val_examples, name='val')

import os
os.makedirs(OUT, exist_ok=True)
model.fit(
    train_dataloader=train_loader,
    evaluator=val_evaluator,
    evaluation_steps=500,
    epochs=2,
    warmup_steps=500,
    optimizer_params={'lr': 2e-5},
    output_path=OUT,
    save_best_model=True,
    show_progress_bar=True,
)
print(f'trained model saved to {OUT}')

In [ ]:
# 7) Post-training eval: AUC + accuracy on val.
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score
model = CrossEncoder(OUT, max_length=384, device='cuda' if torch.cuda.is_available() else 'cpu')
scores = model.predict([[r.text_a, r.text_b] for r in val_df.itertuples(index=False)],
                       batch_size=128, show_progress_bar=True)
labels = val_df['label'].values
auc = roc_auc_score(labels, scores)
# Threshold at 0.5
from scipy.special import expit
preds = (expit(scores) > 0.5).astype(int)
acc = accuracy_score(labels, preds)
print(f'val AUC:      {auc:.4f}')
print(f'val accuracy: {acc:.4f}  (threshold=0.5)')
print(f'positive rate in val: {labels.mean():.4f}')
# Per-class rate
for lab in [0, 1]:
    m = labels == lab
    print(f'  mean score label={lab}: {scores[m].mean():.4f}  (n={m.sum()})')

In [ ]:
# 8) Zip + copy to Drive for local pickup.
import shutil
shutil.make_archive('/content/reward_model', 'zip', OUT)
!ls -lh /content/reward_model.zip

from google.colab import drive
import os
drive.mount('/content/drive')
dst = '/content/drive/MyDrive/recsys2026-reward-model'
os.makedirs(dst, exist_ok=True)
shutil.copy('/content/reward_model.zip', dst)
print(f'\nsaved to: {dst}')
!ls -lh {dst}

In [ ]:
# 9) Optional: browser download.
from google.colab import files
files.download('/content/reward_model.zip')

## Local pickup

```bash
cd /Users/orrimoch/PythonProjs/recsys2026
mkdir -p models/reward_model
unzip -o ~/Downloads/reward_model.zip -d models/reward_model/
# (or copy from Drive if you used that path)
```

Then reference it from `mcrs/response_rerankers/reward_reranker.py` via `REWARD_MODEL_PATH=models/reward_model`.

## Next step: inference-time response reranker

The reward model is only useful combined with multi-candidate response sampling. Build `mcrs/response_rerankers/reward_reranker.py` that:
- Generates K=3-5 responses per query via temperature sampling (T=0.3, T=0.7, T=1.0 — diverse mix).
- Scores each (context, response) pair with the reward model.
- Returns the highest-scoring response.

And `CRS_BASELINE.batch_chat` grows a `response_reranker` slot that plugs in this module when `yaml.response_reranker_type: reward_model` is set.

Expected Blind-A impact: +0.1-0.3 LLM judge if the reward model correlates with Gemini's preferences. Lower bound if AUC < 0.65 on val (noisy signal). Upper bound if AUC > 0.75 (strong signal).